## 1. Install Dependencies

In [1]:
!pip install "datasets<3.0" --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 14.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.


## 2. Install GLiNER & Evaluation Tools

In [2]:
!pip install gliner seqeval datasets --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.8/207.8 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 54.7 MB/s eta 0:00:00


## 3. Import Required Libraries


In [3]:
import torch
from gliner import GLiNER
from datasets import load_dataset
from seqeval.metrics import classification_report

## 4. Check GPU Availability

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## 5. Helper Function: Text Reconstruction

In [5]:
def reconstruct_text_and_spans(tokens):
    text = ""
    spans = []
    for tok in tokens:
        start = len(text)
        text += tok
        end = len(text)
        spans.append((start, end))
        text += " "
    return text.strip(), spans


def bio_tags_from_ids(tag_ids, id2label):
    return [id2label[i] for i in tag_ids]


def align_predictions_to_tokens(predicted_entities, token_spans, label_map, overlap_threshold=0.5):
    tags = ["O"] * len(token_spans)

    for ent in predicted_entities:
        mapped_label = label_map.get(ent["label"])
        if mapped_label is None:
            continue

        first_token_in_span = True
        for i, (tok_start, tok_end) in enumerate(token_spans):
            overlap = max(0, min(ent["end"], tok_end) - max(ent["start"], tok_start))
            tok_len = tok_end - tok_start
            if tok_len > 0 and (overlap / tok_len) >= overlap_threshold:
                tags[i] = f"B-{mapped_label}" if first_token_in_span else f"I-{mapped_label}"
                first_token_in_span = False

    return tags

## 6. Helper Function: GLiNER Prediction

In [6]:
def run_gliner(gliner_model, text, labels):
    entities = gliner_model.predict_entities(text, labels)
    return [
        {"text": ent["text"], "label": ent["label"], "start": ent["start"], "end": ent["end"]}
        for ent in entities
    ]

## 7. Helper Function: Benchmark Evaluation

In [7]:
def benchmark_gliner(model_name, gliner_model, gliner_labels, gold_dataset,
                      gold_id2label, label_map, max_examples=200):
    y_true, y_pred = [], []

    n = min(max_examples, len(gold_dataset))
    for i in range(n):
        example = gold_dataset[i]
        tokens = example["tokens"]
        gold_tags = bio_tags_from_ids(example["tags"], gold_id2label)

        text, token_spans = reconstruct_text_and_spans(tokens)
        predicted_entities = run_gliner(gliner_model, text, gliner_labels)
        pred_tags = align_predictions_to_tokens(predicted_entities, token_spans, label_map)

        y_true.append(gold_tags)
        y_pred.append(pred_tags)

    print(f"\n{'=' * 80}\n{model_name}  (evaluated on {n} examples)\n{'=' * 80}")
    print(classification_report(y_true, y_pred, digits=3))

## 8. Load Gold Standard Datasets

In [8]:
print("Loading gold datasets (tner/bc5cdr, tner/bionlp2004)...", flush=True)
try:
    bc5cdr_gold = load_dataset("tner/bc5cdr", split="test")
    bionlp2004_gold = load_dataset("tner/bionlp2004", split="test")
except RuntimeError as e:
    if "Dataset scripts are no longer supported" in str(e):
        print("Script-based loader blocked by current `datasets` version — "
              "falling back to the auto-converted parquet branch instead.", flush=True)
        bc5cdr_gold = load_dataset(
            "tner/bc5cdr", split="test", revision="refs/convert/parquet"
        )
        bionlp2004_gold = load_dataset(
            "tner/bionlp2004", split="test", revision="refs/convert/parquet"
        )
    else:
        raise

bc5cdr_id2label = {
    0: "O", 1: "B-Chemical", 2: "B-Disease", 3: "I-Disease", 4: "I-Chemical",
}
bionlp2004_id2label = {
    0: "O", 1: "B-DNA", 2: "I-DNA", 3: "B-protein", 4: "I-protein",
    5: "B-cell_type", 6: "I-cell_type", 7: "B-cell_line", 8: "I-cell_line",
    9: "B-RNA", 10: "I-RNA",
}
print("Gold datasets loaded.", flush=True)

Loading gold datasets (tner/bc5cdr, tner/bionlp2004)...


Generating train split:   0%|          | 0/5228 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5330 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5865 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/16619 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1927 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3856 [00:00<?, ? examples/s]

Gold datasets loaded.


## 9. Define Entity Label Mappings

In [9]:
BC5CDR_GLINER_LABELS = ["disease", "chemical"]
BC5CDR_LABEL_MAP_GLINER = {"disease": "Disease", "chemical": "Chemical"}

BIONLP2004_GLINER_LABELS = ["DNA", "RNA", "protein", "cell type", "cell line"]
BIONLP2004_LABEL_MAP_GLINER = {
    "DNA": "DNA", "RNA": "RNA", "protein": "protein",
    "cell type": "cell_type", "cell line": "cell_line",
}

## 10. Load GLiNER-BioMed Model

In [10]:
print("Loading GLiNER-BioMed (uni-encoder — verified compatible with "
      "current gliner library versions; the -bi- variants need a newer "
      "gliner release, see earlier troubleshooting)...", flush=True)
gliner_model = GLiNER.from_pretrained("Ihor/gliner-biomed-large-v1.0")
gliner_model = gliner_model.to(device)
print("  -> done", flush=True)

Loading GLiNER-BioMed (uni-encoder — verified compatible with current gliner library versions; the -bi- variants need a newer gliner release, see earlier troubleshooting)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

  -> done


## 11. Run BC5CDR Benchmark

In [11]:
MAX_EXAMPLES = 200  # raise for a more thorough (slower) run

benchmark_gliner(
    "GLiNER-BioMed (zero-shot) vs gold BC5CDR",
    gliner_model, BC5CDR_GLINER_LABELS,
    bc5cdr_gold, bc5cdr_id2label, BC5CDR_LABEL_MAP_GLINER, MAX_EXAMPLES,
)


GLiNER-BioMed (zero-shot) vs gold BC5CDR  (evaluated on 200 examples)
              precision    recall  f1-score   support

    Chemical      0.715     0.820     0.764       150
     Disease      0.490     0.603     0.541       121

   micro avg      0.611     0.723     0.662       271
   macro avg      0.603     0.712     0.652       271
weighted avg      0.615     0.723     0.664       271



## 12. Run BioNLP2004 Benchmark

In [12]:
benchmark_gliner(
    "GLiNER-BioMed (zero-shot) vs gold BioNLP2004/JNLPBA",
    gliner_model, BIONLP2004_GLINER_LABELS,
    bionlp2004_gold, bionlp2004_id2label, BIONLP2004_LABEL_MAP_GLINER, MAX_EXAMPLES,
)




GLiNER-BioMed (zero-shot) vs gold BioNLP2004/JNLPBA  (evaluated on 200 examples)
              precision    recall  f1-score   support

         DNA      0.633     0.562     0.595        89
         RNA      0.024     1.000     0.047         1
   cell_line      0.357     0.417     0.385        36
   cell_type      0.496     0.759     0.600        83
     protein      0.536     0.721     0.615       197

   micro avg      0.488     0.667     0.564       406
   macro avg      0.409     0.692     0.448       406
weighted avg      0.532     0.667     0.586       406

